In [ ]:
import os
import pandas as pd
from tqdm import tqdm

In [ ]:
val_fold = 9
test_fold = 10

ptbxl_data = "/opt/gpudata/ecg/ptb-xl"
subset_root = "/opt/gpudata/ecg/temp" # path to make subset directories

In [ ]:
df = pd.read_csv(os.path.join(ptbxl_data, "ptbxl_database.csv"), index_col="ecg_id")

In [ ]:
df["strat_fold"].value_counts()

In [ ]:
def make_train_subset(df: pd.DataFrame, train_folds: list[int]):
    # make subsets of training data
    subset = df[df["strat_fold"].isin(train_folds + [val_fold, test_fold])]
    train_size = subset["strat_fold"].value_counts().loc[train_folds].sum()

    train_size_k = train_size // 1024
    if train_size_k == 0:
        train_size_k = str(train_size)
    else:
        train_size_k = f"{train_size_k}k"
    subset_path = os.path.join(subset_root, f"ptb-xl-{train_size_k}")
    os.makedirs(subset_path, exist_ok=True)

    subset.to_csv(os.path.join(subset_path, "ptbxl_database.csv"))

    # also link source waveform data
    for p in [
        "records100",
        "records500",
    ]:
        os.symlink(
            src=os.path.join(ptbxl_data, p),
            dst=os.path.join(subset_path, p),
        )

In [ ]:
for train_folds in tqdm(
    [
        [1, 2, 3, 4],
        [1, 2],
        [1],
    ]
):
    make_train_subset(df, train_folds)